<p align="center">
  <h1 align="center">🍳 Cookbook 03: RecSys & GNN Health Monitoring</h1>
  <p align="center">
    <strong>Diagnosing Embedding Collapse & GNN Over-Smoothing with GradTracer</strong>
  </p>
</p>

---

This recipe targets large-scale recommendation systems using Factorization Machines and Graph Neural Networks.

We will demonstrate how GradTracer identifies critical phenomenon and measure the **performance overhead** of tracking a massive 1 Million item embedding table.

## 1. Setup & MovieLens-1M Loading

In [ ]:
# !pip install gradtracer torch pandas numpy scipy requests tqdm

In [ ]:
import os, pd, numpy as np, torch, torch.nn as nn, time
from torch.utils.data import Dataset, DataLoader
import urllib.request, zipfile
from gradtracer import FlowManager, EmbeddingTracker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Checking for MovieLens-1M...")
if not os.path.exists("ml-1m"):
    url = "http://files.grouplens.org/datasets/movielens/ml-1m.zip"
    urllib.request.urlretrieve(url, "ml-1m.zip")
    with zipfile.ZipFile("ml-1m.zip", 'r') as zip_ref: zip_ref.extractall(".")

columns = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('ml-1m/ratings.dat', sep='::', names=columns, engine='python', encoding='latin-1')
df = df[df['rating'] >= 4].copy()
df['user_id'] = df['user_id'].astype('category').cat.codes
df['item_id'] = df['item_id'].astype('category').cat.codes
num_users, num_items = df['user_id'].nunique(), df['item_id'].max() + 1

## 2. Tracking Matrix Factorization & Diagnostic Report

In [ ]:
class FactorizationMachine(nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        
    def forward(self, user, item):
        return (self.user_emb(user) * self.item_emb(item)).sum(1)

model_fm = FactorizationMachine(num_users, num_items).to(device)
manager = FlowManager()
manager.add_tracker("user", EmbeddingTracker(model_fm.user_emb, name="U_Emb", auto_fix=True))

print("Simulating Training... (manager.step() will track embedding drift)")
optimizer = torch.optim.Adam(model_fm.parameters(), lr=0.1)
for _ in range(20):
    u = torch.randint(0, num_users, (1024,), device=device)
    i = torch.randint(0, num_items, (1024,), device=device)
    optimizer.zero_grad()
    loss = nn.BCEWithLogitsLoss()(model_fm(u, i), torch.ones(1024, device=device))
    loss.backward()
    manager.step()
    optimizer.step()

print("\n" + "="*60)
print("📈 RecSys Embedding Health Audit Result")
print("="*60)
# 🔧 NEW: Showing the powerful diagnostic output of EmbeddingTracker
manager.get_tracker("user").report()

## 3. [Bonus] Performance Benchmark (1M+ Embeddings)
Proving that GradTracer adds minimal overhead even on production-scale embedding tables.

In [ ]:
LARGE_NUM = 1_000_000
DIM = 64
large_emb = nn.Embedding(LARGE_NUM, DIM).to(device)
optimizer = torch.optim.Adam(large_emb.parameters(), lr=0.01)
indices = torch.randint(0, LARGE_NUM, (1024,), device=device)

def run_bench(use_tracker=False):
    gt = None
    if use_tracker: gt = EmbeddingTracker(large_emb, track_interval=100)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(200):
        optimizer.zero_grad()
        # Sparse access pattern
        loss = large_emb(indices).sum()
        loss.backward()
        if use_tracker: gt.step()
        optimizer.step()
    torch.cuda.synchronize()
    return time.time() - start

t_base = run_bench(False)
t_gt = run_bench(True)
overhead = (t_gt - t_base) / t_base * 100

print(f"\n{'Scenario':<25} | {'Duration (s)':<15} | {'Overhead'}")
print("-"*60)
print(f"{'Native PyTorch':<25} | {t_base:15.3f} | {'0.00%'}")
print(f"{'GradTracer (1M Embed)':<25} | {t_gt:15.3f} | {overhead:8.2f}%")
print("-"*60)
if overhead < 5.0:
    print("✅ SUCCESS: Performance overhead is within < 5% production limits.")